# 🌿 HUẤN LUYỆN & ĐÁNH GIÁ MÔ HÌNH NHẬN DIỆN 90 LOÀI KIỂNG LÁ (SEED 999)
### 📌 Bộ dữ liệu: 30.000 ảnh - Đã làm sạch trùng lặp bằng thuật toán Perceptual Hash (pHash) & MD5
**Cấu hình thực nghiệm chuẩn học thuật:**
- **Phân chia dữ liệu:** Stratified Split `80% Train` – `10% Val` – `10% Test`.
- **Random Seed:** `999` (Hoàn thành bộ 3 thực nghiệm độc lập: Seed 42, Seed 123, Seed 999).
- **Kiến trúc:** MobileNetV2 Transfer Learning 2 giai đoạn (Head + Fine-tuning 30 lớp).
- **Đầu ra học thuật:** Ma trận nhầm lẫn 90x90, Metrics chuẩn, Bảng tổng hợp Mean ± Std và mã LaTeX chèn thẳng vào báo cáo Overleaf.

In [ ]:
# ==============================================================================
# === BƯỚC 1: KẾT NỐI DRIVE, THIẾT LẬP SEED 999 & TỰ ĐỘNG TÌM FILE DỮ LIỆU ===
# ==============================================================================
import os
import json
import random
import shutil
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from google.colab import drive

# 1. Kết nối Google Drive
if not Path('/content/drive/MyDrive').exists():
    print('🔗 Đang kết nối Google Drive...')
    drive.mount('/content/drive')

# 2. Cố định Seed 999 cho tính tái lập thực nghiệm (Reproducibility)
RANDOM_SEED = 999
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

# 3. Tự động dò tìm file zip 6.7 GB (90 loài)
PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
candidate_paths = [
    PROJECT / '01_Du_Lieu_30k_Va_Labels' / 'merged_v1' / 'merged_90_30k_chuan.zip',
    PROJECT / '01_Du_Lieu_Goc' / 'merged_90_30k_chuan.zip',
    PROJECT / 'merged_v1' / 'merged_90_30k_chuan.zip'
]

ZIP_30K_PATH = None
for p in candidate_paths:
    if p.is_file():
        ZIP_30K_PATH = p
        break

if ZIP_30K_PATH is None:
    found = list(PROJECT.rglob('*merged_90_30k*.zip'))
    if found:
        ZIP_30K_PATH = found[0]

assert ZIP_30K_PATH is not None and ZIP_30K_PATH.is_file(), f'❌ Không tìm thấy file zip 30k trong {PROJECT}!'

# 4. Cấu hình thư mục lưu kết quả riêng biệt cho Seed 999
RAW_30K_DIR = Path('/content/raw_30k_extracted')
DATA_30K_SPLIT = Path('/content/data_30k_split_80_10_10')
SAVE_30K_ROOT = PROJECT / 'train_90_lop_30k_seed_999'
SAVE_30K_ROOT.mkdir(parents=True, exist_ok=True)

print('=' * 68)
print('🚀 MÔI TRƯỜNG HUẤN LUYỆN SEED 999 ĐÃ SẴN SÀNG:')
print('  • TensorFlow version :', tf.__version__)
print('  • GPU Devices        :', tf.config.list_physical_devices('GPU'))
print(f'  • File dữ liệu 30k   : {ZIP_30K_PATH} ({ZIP_30K_PATH.stat().st_size / (1024**2):.1f} MB)')
print(f'  • Thư mục lưu kết quả: {SAVE_30K_ROOT}')
print('=' * 68)


In [ ]:
# ==============================================================================
# === BƯỚC 2: GIẢI NÉN VÀ PHÂN CHIA STRATIFIED SPLIT 80% - 10% - 10%         ===
# ==============================================================================
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# 1. Dọn dẹp thư mục tạm cũ nếu trước đó giải nén sai số lớp
if RAW_30K_DIR.exists():
    sub_dirs = [d for d in RAW_30K_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')]
    if len(sub_dirs) != 90:
        print('🧹 Thư mục tạm cũ không chuẩn 90 lớp, đang xóa để giải nén lại...')
        shutil.rmtree(RAW_30K_DIR)

# 2. Giải nén dữ liệu chuẩn 6.7 GB
if not RAW_30K_DIR.exists() or len(list(RAW_30K_DIR.glob('*'))) == 0:
    RAW_30K_DIR.mkdir(parents=True, exist_ok=True)
    print(f'📦 Đang giải nén bộ dữ liệu 30.000 ảnh từ: {ZIP_30K_PATH.name} (mất 1-2 phút)...')
    !unzip -q "{ZIP_30K_PATH}" -d /content/raw_30k_extracted
    print('✅ Giải nén hoàn tất!')

# 3. Quét gom toàn bộ ảnh của 90 loài
class_images = {}
for root, dirs, files in os.walk(str(RAW_30K_DIR)):
    folder_name = os.path.basename(root)
    valid_f = [Path(root) / f for f in files if Path(f).suffix.lower() in EXTENSIONS]
    if valid_f and not folder_name.startswith('.'):
        if folder_name not in class_images:
            class_images[folder_name] = []
        class_images[folder_name].extend(valid_f)

class_folders = sorted(list(class_images.keys()))
print(f'🎯 Đã xác định chính xác: {len(class_folders)} loài cây.')
assert len(class_folders) == 90, f'Lỗi: Phát hiện {len(class_folders)} loài (khác 90)!'

# 4. Phân chia Stratified Split theo tỷ lệ 80 - 10 - 10 với SEED 999
if DATA_30K_SPLIT.exists():
    shutil.rmtree(DATA_30K_SPLIT)

for split in ['train', 'val', 'test']:
    for c in class_folders:
        (DATA_30K_SPLIT / split / c).mkdir(parents=True, exist_ok=True)

rng = random.Random(RANDOM_SEED)
split_rows = []

print(f'✂️ Đang phân chia ảnh theo RANDOM_SEED = {RANDOM_SEED}...')
for cls_name in class_folders:
    all_imgs = sorted(list(set(class_images[cls_name])), key=lambda p: p.name)
    rng.shuffle(all_imgs)

    n = len(all_imgs)
    n_train = int(n * 0.80)
    n_val = int(n * 0.10)
    if n_val == 0 and n >= 3:
        n_val = 1

    splits = {
        'train': all_imgs[:n_train],
        'val': all_imgs[n_train:n_train + n_val],
        'test': all_imgs[n_train + n_val:]
    }

    for s_name, paths in splits.items():
        dest_d = DATA_30K_SPLIT / s_name / cls_name
        for p_src in paths:
            p_dst = dest_d / p_src.name
            try:
                os.link(p_src, p_dst)
            except OSError:
                shutil.copy2(p_src, p_dst)

    split_rows.append({
        'label': cls_name,
        'train': len(splits['train']),
        'val': len(splits['val']),
        'test': len(splits['test']),
        'total': n
    })

split_df = pd.DataFrame(split_rows)
split_df.to_csv(SAVE_30K_ROOT / 'split_counts_30k_80_10_10.csv', index=False, encoding='utf-8-sig')

tot_train = int(split_df['train'].sum())
tot_val = int(split_df['val'].sum())
tot_test = int(split_df['test'].sum())
tot_all = int(split_df['total'].sum())

print('=' * 68)
print('📊 KẾT QUẢ PHÂN CHIA STRATIFIED SPLIT 80 - 10 - 10 (SEED 999):')
print(f'  • Tổng số ảnh toàn bộ : {tot_all:,} ảnh (100%)')
print(f'  • Tập Train (80%)      : {tot_train:,} ảnh ({tot_train/tot_all:.2%})')
print(f'  • Tập Val   (10%)      : {tot_val:,} ảnh ({tot_val/tot_all:.2%})')
print(f'  • Tập Test  (10%)      : {tot_test:,} ảnh ({tot_test/tot_all:.2%}) -> Hoàn toàn độc lập')
print('=' * 68)


In [ ]:
# ==============================================================================
# === BƯỚC 3: DATA PIPELINE TENSORFLOW & TÍNH BALANCED CLASS WEIGHTS         ===
# ==============================================================================
from sklearn.utils.class_weight import compute_class_weight

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

def load_dataset(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        str(DATA_30K_SPLIT / split),
        image_size=IMAGE_SIZE,
        interpolation='nearest',
        batch_size=BATCH_SIZE,
        label_mode='int',
        shuffle=shuffle,
        seed=RANDOM_SEED
    )

train_ds_raw = load_dataset('train', True)
val_ds_raw = load_dataset('val', False)
test_ds_raw = load_dataset('test', False)

class_names = train_ds_raw.class_names
assert len(class_names) == 90, f'Số lớp không đúng 90: {len(class_names)}'

# Tính trọng số cân bằng lớp (Balanced Class Weights)
y_train_all = np.concatenate([y.numpy() for _, y in train_ds_raw], axis=0)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=y_train_all
)
class_weight_dict = {i: float(w) for i, w in enumerate(weights)}

train_ds = train_ds_raw.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.prefetch(tf.data.AUTOTUNE)

HEAD_DIR = SAVE_30K_ROOT / 'head'
FINE_DIR = SAVE_30K_ROOT / 'finetune'
HEAD_DIR.mkdir(parents=True, exist_ok=True)
FINE_DIR.mkdir(parents=True, exist_ok=True)

(SAVE_30K_ROOT / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
(HEAD_DIR / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
(FINE_DIR / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'✅ Đã chuẩn bị xong Data Pipeline với {len(y_train_all):,} ảnh Train và {len(class_names)} lớp!')


In [ ]:
# ==============================================================================
# === BƯỚC 4: HUẤN LUYỆN GIAI ĐOẠN 1 (TRANSFER LEARNING - ĐÓNG BĂNG BACKBONE) ===
# ==============================================================================
import math
from tensorflow.keras import layers

HEAD_STATE = HEAD_DIR / 'progress.json'
HEAD_BEST = HEAD_DIR / 'best_model.keras'

class CustomCheckpoint(keras.callbacks.Callback):
    def __init__(self, save_dir, state_path, best_path, patience=5, min_delta=0.001):
        super().__init__()
        self.save_dir = save_dir
        self.state_path = state_path
        self.best_path = best_path
        self.patience = patience
        self.min_delta = min_delta
        if self.state_path.exists():
            self.state = json.loads(self.state_path.read_text(encoding='utf-8'))
        else:
            self.state = {'next_epoch': 0, 'best_loss': None, 'reference_loss': None, 'wait': 0, 'stopped': False}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = float(logs['val_loss'])
        if not math.isfinite(loss):
            self.model.stop_training = True
            return
        if self.state['best_loss'] is None or loss < self.state['best_loss']:
            temp = self.save_dir / 'best_temp.keras'
            self.model.save(str(temp))
            os.replace(temp, self.best_path)
            self.state['best_loss'] = loss
        ref = self.state['reference_loss']
        if ref is None or (ref - loss) > self.min_delta:
            self.state['reference_loss'] = loss
            self.state['wait'] = 0
        else:
            self.state['wait'] += 1
            if self.state['wait'] >= self.patience:
                print(f'Dừng sớm (Early Stop) tại epoch {epoch+1}')
                self.state['stopped'] = True
                self.model.stop_training = True
        self.state['next_epoch'] = epoch + 1
        self.state_path.write_text(json.dumps(self.state, indent=2), encoding='utf-8')

save_head_cb = CustomCheckpoint(HEAD_DIR, HEAD_STATE, HEAD_BEST, patience=5)

if save_head_cb.state['stopped'] and HEAD_BEST.exists():
    print('✅ Giai đoạn 1 (Head) đã hoàn tất trước đó.')
else:
    base_model = keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = layers.RandomFlip('horizontal', seed=RANDOM_SEED)(inputs)
    x = layers.RandomRotation(0.08, seed=RANDOM_SEED)(x)
    x = layers.RandomZoom(0.10, seed=RANDOM_SEED)(x)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3, seed=RANDOM_SEED)(x)
    outputs = layers.Dense(len(class_names), activation='softmax', name='pred_90')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')]
    )
    print('🚀 BẮT ĐẦU HUẤN LUYỆN GIAI ĐOẠN 1 (ĐÓNG BĂNG BACKBONE)...')
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=25,
        class_weight=class_weight_dict,
        callbacks=[save_head_cb, keras.callbacks.CSVLogger(str(HEAD_DIR / 'history.csv'))]
    )


In [ ]:
# ==============================================================================
# === BƯỚC 5: HUẤN LUYỆN GIAI ĐOẠN 2 (FINE-TUNING 30 LỚP CUỐI MOBILENETV2)     ===
# ==============================================================================
FINE_STATE = FINE_DIR / 'progress.json'
FINE_BEST = FINE_DIR / 'best_model.keras'

save_fine_cb = CustomCheckpoint(FINE_DIR, FINE_STATE, FINE_BEST, patience=5)

if save_fine_cb.state['stopped'] and FINE_BEST.exists():
    print('✅ Giai đoạn 2 (Fine-tuning) đã hoàn tất trước đó.')
else:
    model = keras.models.load_model(str(HEAD_BEST))
    base_model = next(
        layer for layer in model.layers
        if isinstance(layer, keras.Model) and 'mobilenet' in layer.name.lower()
    )
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        layer.trainable = not isinstance(layer, layers.BatchNormalization)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')]
    )
    print('🚀 BẮT ĐẦU FINE-TUNING 30 LỚP CUỐI (LR = 1e-5)...')
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=25,
        class_weight=class_weight_dict,
        callbacks=[save_fine_cb, keras.callbacks.CSVLogger(str(FINE_DIR / 'history.csv'))]
    )


In [ ]:
# ==============================================================================
# === BƯỚC 6: ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP & XUẤT MA TRẬN NHẦM LẪN (SEED 999) ===
# ==============================================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

EVAL_DIR = SAVE_30K_ROOT / 'evaluation_report_30k_80_10_10'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

best_model = keras.models.load_model(str(FINE_BEST), compile=False)

print('🔍 Đang kiểm thử trên tập Test độc lập 100%...')
true_batches, score_batches = [], []
for images, labels in test_ds:
    scores = best_model(images, training=False).numpy()
    true_batches.append(labels.numpy())
    score_batches.append(scores)

y_true = np.concatenate(true_batches)
scores_all = np.concatenate(score_batches)
y_pred = scores_all.argmax(axis=1)

top1_acc = float(accuracy_score(y_true, y_pred))
top3_acc = float(np.mean(np.any(np.argsort(scores_all, axis=1)[:, -3:] == y_true[:, None], axis=1)))
macro_prec = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
macro_rec = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
weighted_prec = float(precision_score(y_true, y_pred, average='weighted', zero_division=0))
weighted_rec = float(recall_score(y_true, y_pred, average='weighted', zero_division=0))
weighted_f1 = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))

metrics_dict = {
    'seed': RANDOM_SEED,
    'dataset_version': '30.000 images (pHash cleaned)',
    'split_ratio': '80-10-10',
    'test_images': int(len(y_true)),
    'correct_predictions': int(np.sum(y_true == y_pred)),
    'incorrect_predictions': int(np.sum(y_true != y_pred)),
    'top1_accuracy': top1_acc,
    'top3_accuracy': top3_acc,
    'macro_precision': macro_prec,
    'macro_recall': macro_rec,
    'macro_f1': macro_f1,
    'weighted_precision': weighted_prec,
    'weighted_recall': weighted_rec,
    'weighted_f1': weighted_f1
}

(EVAL_DIR / 'metrics.json').write_text(json.dumps(metrics_dict, indent=2), encoding='utf-8')
cm = confusion_matrix(y_true, y_pred, labels=np.arange(90))
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(EVAL_DIR / 'confusion_matrix.csv')

rep_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
pd.DataFrame(rep_dict).transpose().to_csv(EVAL_DIR / 'classification_report.csv')

print('=' * 68)
print(f'🎉 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST (SEED {RANDOM_SEED}):')
print('=' * 68)
print(f'  • Tổng số ảnh kiểm thử (Test): {len(y_true):,} ảnh')
print(f'  • Top-1 Accuracy             : {top1_acc * 100:.2f}%')
print(f'  • Top-3 Accuracy             : {top3_acc * 100:.2f}%')
print(f'  • Weighted Precision         : {weighted_prec * 100:.2f}%')
print(f'  • Weighted Recall            : {weighted_rec * 100:.2f}%')
print(f'  • Weighted F1-Score          : {weighted_f1 * 100:.2f}%')
print('=' * 68)
print(f'📁 Đã lưu kết quả tại: {EVAL_DIR}')


In [ ]:
# ==============================================================================
# === BƯỚC 7: TỔNG HỢP 3 SEEDS (42, 123, 999) & XUẤT BẢNG MEAN ± STD (LATEX) ===
# ==============================================================================
print('📊 ĐANG TỔNG HỢP KẾT QUẢ CẢ 3 SEEDS ĐỘC LẬP...')

seeds_to_check = [42, 123, 999]
metrics_all = {}
reports_all = {}

for s in seeds_to_check:
    possible_dirs = [
        PROJECT / f'train_90_lop_30k_seed_{s}' / 'evaluation_report_30k_80_10_10',
        PROJECT / f'train_90_lop_seed_{s}' / f'evaluation_report_seed_{s}',
        PROJECT / 'train_90_lop_30k_anh' / 'evaluation_report_30k_80_10_10'
    ]
    for d in possible_dirs:
        m_file = d / 'metrics.json'
        r_file = d / 'classification_report.csv'
        if m_file.exists():
            metrics_all[s] = json.loads(m_file.read_text(encoding='utf-8'))
            if r_file.exists():
                reports_all[s] = pd.read_csv(r_file, index_col=0)
            break

print(f'✅ Tìm thấy dữ liệu của {len(metrics_all)}/3 seeds: {list(metrics_all.keys())}')

if len(metrics_all) > 0:
    def fmt_stat(vals, pct=True):
        arr = np.array(vals, dtype=float)
        if pct:
            arr = arr * 100.0
        m = np.mean(arr)
        s = np.std(arr, ddof=1) if len(arr) > 1 else 0.0
        return f"{m:.2f} ± {s:.2f}%" if pct else f"{m:.4f} ± {s:.4f}", f"${m:.2f} \\pm {s:.2f}$"

    f_seeds = sorted(list(metrics_all.keys()))
    top1_vals = [metrics_all[s]['top1_accuracy'] for s in f_seeds]
    top3_vals = [metrics_all[s]['top3_accuracy'] for s in f_seeds]
    p_vals = [metrics_all[s]['weighted_precision'] for s in f_seeds]
    r_vals = [metrics_all[s]['weighted_recall'] for s in f_seeds]
    f1_vals = [metrics_all[s]['weighted_f1'] for s in f_seeds]

    summary_rows = []
    for name, v in [('Top-1 Accuracy', top1_vals), ('Top-3 Accuracy', top3_vals), ('Weighted Precision', p_vals), ('Weighted Recall', r_vals), ('Weighted F1-Score', f1_vals)]:
        text_val, latex_val = fmt_stat(v)
        row = {'Chỉ số (Metric)': name, 'Giá trị (Mean ± Std)': text_val, 'Mã LaTeX (Overleaf)': latex_val}
        for s in f_seeds:
            row[f'Seed {s}'] = f"{metrics_all[s]['top1_accuracy' if 'top-1' in name.lower() else 'top3_accuracy' if 'top-3' in name.lower() else 'weighted_precision' if 'precision' in name.lower() else 'weighted_recall' if 'recall' in name.lower() else 'weighted_f1']*100:.2f}%"
        summary_rows.append(row)

    df_summary = pd.DataFrame(summary_rows)
    display(df_summary)
    out_excel = PROJECT / 'Ket_Qua_Tong_Hop_3_Seeds_Overleaf.xlsx'
    df_summary.to_excel(out_excel, index=False)
    print(f'📁 Đã xuất bảng tổng hợp Mean ± Std: {out_excel}')
